# Marketing ROI Optimisation — Exploratory Data Analysis

**Project**: EFREI M1 Data Engineering — RNCP40875  
**Dataset**: `data/marketing_labelled.csv` (4 572 campaigns)  
**Target**: `Sales` (regression) / `perf_class` (classification: Low / Medium / High)

---
## Contents
1. Setup & Data Loading  
2. Dataset Overview  
3. Univariate Distributions  
4. Correlation Analysis  
5. Sales by Influencer Tier  
6. Budget Breakdown  
7. Performance Class Distribution  
8. Key Findings

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import load_data, NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET_REG, TARGET_CLF

%matplotlib inline
sns.set_theme(style="whitegrid", palette="steelblue")
plt.rcParams["figure.dpi"] = 100

df = load_data()
print(f"Shape: {df.shape}")
df.head()

## 2. Dataset Overview

In [ ]:
print("=== dtypes ===")
print(df.dtypes)
print("\n=== Missing values ===")
print(df.isnull().sum())
print("\n=== Descriptive statistics ===")
df.describe().round(2)

## 3. Univariate Distributions

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cols = NUMERIC_FEATURES + [TARGET_REG]
for ax, col in zip(axes, cols):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color="steelblue")
    ax.set_title(col)
plt.suptitle("Univariate Distributions", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
corr_cols = NUMERIC_FEATURES + [TARGET_REG]
corr = df[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[0])
axes[0].set_title("Correlation Matrix")

# Bar: correlation with Sales
sales_corr = corr[TARGET_REG].drop(TARGET_REG).sort_values(ascending=False)
axes[1].bar(sales_corr.index, sales_corr.values, color="steelblue", edgecolor="k")
axes[1].set_title("Correlation with Sales")
axes[1].set_ylabel("Pearson r")
plt.tight_layout()
plt.show()

print("Correlations with Sales:")
print(sales_corr.to_string())

## 5. Sales by Influencer Tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

order = ["Mega", "Macro", "Micro", "Nano"]
sns.boxplot(data=df, x="Influencer", y=TARGET_REG, order=order,
            palette="Set2", ax=axes[0])
axes[0].set_title("Sales Distribution by Influencer")

mean_sales = df.groupby("Influencer")[TARGET_REG].mean().reindex(order)
axes[1].bar(mean_sales.index, mean_sales.values, color="steelblue", edgecolor="k")
axes[1].set_title("Mean Sales by Influencer")
axes[1].set_ylabel("Mean Sales")

plt.tight_layout()
plt.show()

## 6. Budget Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Average budget per channel
avg_budget = df[NUMERIC_FEATURES].mean()
axes[0].bar(avg_budget.index, avg_budget.values, color=["#4878CF", "#6ACC65", "#D65F5F"], edgecolor="k")
axes[0].set_title("Average Budget per Channel ($M)")
axes[0].set_ylabel("Budget ($M)")

# Scatter: TV vs Sales
axes[1].scatter(df["TV"], df[TARGET_REG], alpha=0.2, s=8, color="steelblue")
axes[1].set_xlabel("TV Budget ($M)")
axes[1].set_ylabel("Sales")
axes[1].set_title("TV Budget vs Sales")

plt.tight_layout()
plt.show()

## 7. Performance Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = df[TARGET_CLF].value_counts().reindex(["Low", "Medium", "High"])
axes[0].bar(counts.index, counts.values, color=["#D65F5F", "#F3A712", "#6ACC65"], edgecolor="k")
axes[0].set_title("Performance Class Counts")
axes[0].set_ylabel("Count")

axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%",
            colors=["#D65F5F", "#F3A712", "#6ACC65"], startangle=90)
axes[1].set_title("Performance Class Distribution")

plt.tight_layout()
plt.show()

print("\nClass distribution:")
print(counts.to_string())

## 8. Key Findings

| Finding | Detail |
|---------|--------|
| **Strongest predictor** | TV budget has the highest Pearson correlation with Sales (~0.78) |
| **Influencer effect** | Mega influencers yield highest mean sales; Nano the lowest |
| **Budget spread** | TV dominates budget allocation; Social Media smallest share |
| **Class balance** | Quantile-based split yields roughly equal Low / Medium / High classes (~33% each) |
| **No severe outliers** | Distributions approximately normal after log-scale inspection |
| **Missing values** | Minimal missing data; median/mode imputation sufficient |

These findings directly motivate the preprocessing choices (StandardScaler, median imputation) and confirm that tree-based models should capture the non-linear interactions between channels effectively.